# 09 — Training C-PPO and D-DQN for Market Making

Trains the two RL agents from `paper_replication.rl` (paper Section III-E) on the BTC/USDT `MarketMakingEnv`: `ContinuousActorCritic` via PPO (the continuous action space, Eq. 8-11) and `DuelingQNetwork` via Dueling DQN (the discrete action space, III-C1). Evaluates both against three baselines (Random, Fixed-spread, an Avellaneda-Stoikov-inspired policy) on a held-out, chronologically-later test split, using the paper's own Table II metrics (ND-PnL, PnLMAP, Profit Ratio, Sharpe).

Builds on notebook 05 (Attn-LOB pretraining) and the RL model built in `paper_replication.rl` (see `docs/rl_model.md`). The Attn-LOB backbone is loaded from that pretraining run's checkpoint and kept **frozen** during RL training — only the small trunk + policy/value (or Q) heads are trained. Since the backbone's output doesn't depend on the agent's actions, it's precomputed once over the whole dataset (`rl.feature_cache.precompute_lob_features`) rather than recomputed every RL step — this is what keeps a real training run inside a laptop-scale compute budget instead of requiring a cluster.

**This is a compute-budgeted run, not paper-scale training** — see `docs/experiments.md` for exactly what that means and what it doesn't.


In [ ]:
from __future__ import annotations

import json
import pathlib
import time

import numpy as np
import pandas as pd
import torch

from paper_replication.features.config import FeatureConfig
from paper_replication.rl.baselines import (
    AvellanedaStoikovPolicy,
    FixedSpreadPolicy,
    RandomPolicy,
)
from paper_replication.rl.config import RLConfig
from paper_replication.rl.dqn import DQNConfig, train_dqn
from paper_replication.rl.env import MarketMakingEnv, Observation
from paper_replication.rl.evaluate import evaluate_policy
from paper_replication.rl.feature_cache import precompute_lob_features
from paper_replication.rl.market_data import build_market_dataset, chronological_split
from paper_replication.rl.networks import (
    ContinuousActorCritic,
    DuelingQNetwork,
    RLNetworkConfig,
)
from paper_replication.rl.ppo import PPOConfig, train_ppo
from paper_replication.training.checkpoint import load_checkpoint

LOB_PATH = "../data/btc_usdt_20221019_20221030_lob.parquet"
ATTN_LOB_CHECKPOINT = "../data/processed/attn_lob_checkpoint_tuned.pt"
PPO_CHECKPOINT = pathlib.Path("../data/processed/rl_ppo_checkpoint.pt")
DQN_CHECKPOINT = pathlib.Path("../data/processed/rl_dqn_checkpoint.pt")
RESULTS_PATH = pathlib.Path("../data/processed/rl_results.json")

print("cuda available:", torch.cuda.is_available())

## Build the market dataset and split it chronologically

Reuses the same LOB snapshot file as the pretraining notebooks. `build_market_dataset` (unlike `build_feature_dataset`) skips the label machinery entirely — it produces LOB-state windows, dynamic-state features, and raw best bid/ask/mid price, which is everything `MarketMakingEnv` needs to execute simulated quotes.

`test_frac=0.2` (vs. the pretraining pipeline's 50/50 train/test split): RL training needs many episodes to learn from, and the held-out test set only needs enough episodes for a stable metrics estimate, not a large fraction of the data.

In [ ]:
fc = FeatureConfig()
ds = build_market_dataset(LOB_PATH, fc)
print(f"dataset: {len(ds)} rows, {len(ds.dynamic_feature_names)} dynamic features")

splits = chronological_split(ds, test_frac=0.2)
print(f"train: {len(splits['train'])} rows, test: {len(splits['test'])} rows")

## Load the pretrained Attn-LOB backbone and precompute its features

`attn_lob_checkpoint_tuned.pt` is Exp 1b's winning pretrain run (`docs/experiments.md`) — the best-performing Attn-LOB checkpoint this replication has trained. `precompute_lob_features` runs it once, in large batches, over the train and test splits' `lob_state` windows on the GPU; RL training then does one array lookup per step instead of one CNN+attention forward pass.

In [ ]:
attn_lob = load_checkpoint(ATTN_LOB_CHECKPOINT)
print("Attn-LOB backbone config:", attn_lob.config)

device_for_cache = "cuda" if torch.cuda.is_available() else "cpu"
t0 = time.time()
train_cache = precompute_lob_features(
    attn_lob, splits["train"].lob_state, device=device_for_cache, batch_size=2048
)
test_cache = precompute_lob_features(
    attn_lob, splits["test"].lob_state, device=device_for_cache, batch_size=2048
)
print(f"feature cache built in {time.time() - t0:.1f}s")

## Build the environments

Separate `MarketMakingEnv` instances for train/test (so episodes never straddle the split boundary) and for each action space (continuous for C-PPO/baselines, discrete for D-DQN). `RLConfig`'s defaults match the paper's own reported hyperparameters where stated (`omega=10`, `eta=0.5`, `zeta=0.01`, `max_bias=0.05`, `max_spread=0.1`); `minimum_trade_unit` and `episode_length` are this replication's documented substitutes (see `docs/rl_model.md`).

In [ ]:
rl_config = RLConfig(
    minimum_trade_unit=0.001,
    omega=10.0,
    max_bias=0.05,
    max_spread=0.1,
    eta=0.5,
    zeta=0.01,
    episode_length=30,
)

train_env_c = MarketMakingEnv(
    splits["train"], rl_config, action_space_kind="continuous"
)
test_env_c = MarketMakingEnv(splits["test"], rl_config, action_space_kind="continuous")
train_env_d = MarketMakingEnv(splits["train"], rl_config, action_space_kind="discrete")
test_env_d = MarketMakingEnv(splits["test"], rl_config, action_space_kind="discrete")

print(
    f"train episodes: {train_env_c.n_episodes}, test episodes: {test_env_c.n_episodes}"
)

net_config = RLNetworkConfig(
    attn_lob_config=attn_lob.config,
    n_dynamic_features=len(ds.dynamic_feature_names),
    n_agent_features=2,
    trunk_hidden_dim=128,
)

## Train C-PPO

Standard clipped-surrogate PPO (`rl.ppo.train_ppo`) over `ContinuousActorCritic`. Trains only the trunk + policy/value heads — the Attn-LOB backbone stays frozen, fed from `train_cache`.

In [ ]:
ppo_model = ContinuousActorCritic(net_config)
ppo_config = PPOConfig(
    n_updates=300,
    episodes_per_update=16,
    update_epochs=4,
    minibatch_size=128,
    learning_rate=3e-4,
    device="cpu",
    seed=0,
)

t0 = time.time()
ppo_history = train_ppo(train_env_c, ppo_model, train_cache, ppo_config)
print(f"C-PPO trained in {time.time() - t0:.1f}s")
print("last update:", ppo_history[-1])

PPO_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {"state_dict": ppo_model.state_dict(), "net_config": net_config}, PPO_CHECKPOINT
)

## Train D-DQN

Standard DQN with a target network and replay buffer (`rl.dqn.train_dqn`) over `DuelingQNetwork` (Wang et al.'s value/advantage decomposition, matching the paper's choice of Dueling DQN).

In [ ]:
dqn_model = DuelingQNetwork(net_config, n_actions=8)
dqn_config = DQNConfig(
    n_steps=150_000,
    warmup_steps=2_000,
    batch_size=128,
    buffer_size=50_000,
    train_interval=4,
    target_update_interval=1_000,
    log_interval=5_000,
    epsilon_decay_steps=100_000,
    learning_rate=1e-3,
    device="cpu",
    seed=0,
)

t0 = time.time()
dqn_history = train_dqn(train_env_d, dqn_model, train_cache, dqn_config)
print(f"D-DQN trained in {time.time() - t0:.1f}s")
print("last log:", dqn_history[-1])

torch.save(
    {"state_dict": dqn_model.state_dict(), "net_config": net_config}, DQN_CHECKPOINT
)

## Evaluate everyone on the held-out test episodes

C-PPO and D-DQN are evaluated **deterministically** — the Beta distribution's mean (`alpha / (alpha + beta)`) for C-PPO, the greedy action for D-DQN — rather than sampling stochastically, since this is an evaluation pass, not further exploration. All seven policies (C-PPO, D-DQN, Random, three Fixed-spread levels, an Avellaneda-Stoikov-inspired policy) run through the exact same `evaluate_policy` harness over every test episode, so the comparison is apples-to-apples.

In [ ]:
def ppo_policy(obs: Observation, row: int) -> np.ndarray:
    lob_t = torch.from_numpy(test_cache[row]).unsqueeze(0)
    dyn_t = torch.from_numpy(obs.dynamic_state.astype(np.float32)).unsqueeze(0)
    agent_t = torch.from_numpy(obs.agent_state.astype(np.float32)).unsqueeze(0)
    with torch.no_grad():
        alpha, beta, _ = ppo_model.forward_from_lob_features(lob_t, dyn_t, agent_t)
        action = (alpha / (alpha + beta)).squeeze(0).numpy()
    return np.asarray(action)


def dqn_policy(obs: Observation, row: int) -> int:
    lob_t = torch.from_numpy(test_cache[row]).unsqueeze(0)
    dyn_t = torch.from_numpy(obs.dynamic_state.astype(np.float32)).unsqueeze(0)
    agent_t = torch.from_numpy(obs.agent_state.astype(np.float32)).unsqueeze(0)
    with torch.no_grad():
        q = dqn_model.forward_from_lob_features(lob_t, dyn_t, agent_t)
        action = int(q.argmax(dim=-1).item())
    return action


rv_idx = ds.dynamic_feature_names.index("rv_300s")
as_policy = AvellanedaStoikovPolicy(
    mid_price=splits["test"].mid_price,
    max_inventory=rl_config.max_inventory,
    max_bias=rl_config.max_bias,
    max_spread=rl_config.max_spread,
    rv_feature_index=rv_idx,
)

policies = [
    ("C-PPO", test_env_c, ppo_policy),
    ("D-DQN", test_env_d, dqn_policy),
    ("Random", test_env_c, RandomPolicy(seed=0)),
    ("Fixed_1", test_env_c, FixedSpreadPolicy(spread_frac=0.15)),
    ("Fixed_2", test_env_c, FixedSpreadPolicy(spread_frac=0.5)),
    ("Fixed_3", test_env_c, FixedSpreadPolicy(spread_frac=1.0)),
    ("AS", test_env_c, as_policy),
]

test_indices = list(range(test_env_c.n_episodes))
rows = []
for name, env, policy in policies:
    m = evaluate_policy(env, policy, test_indices)
    rows.append(
        {
            "policy": name,
            "n_episodes": m.n_episodes,
            "ND-PnL": f"{m.nd_pnl_mean:.3g} +/- {m.nd_pnl_std:.3g}",
            "PnLMAP": f"{m.pnlmap_mean:.3g} +/- {m.pnlmap_std:.3g}",
            "PR": f"{m.profit_ratio_mean:.3g} +/- {m.profit_ratio_std:.3g}",
            "Sharpe": f"{m.sharpe:.3g}",
            "mean_abs_position": f"{m.mean_abs_position:.3g}",
        }
    )

results_df = pd.DataFrame(rows).set_index("policy")
results_df

## Save results

Written to `data/processed/rl_results.json` so `docs/experiments.md` can cite exact numbers without re-running this notebook.

In [ ]:
RESULTS_PATH.write_text(
    json.dumps(
        {
            "results": rows,
            "n_train_episodes": train_env_c.n_episodes,
            "n_test_episodes": test_env_c.n_episodes,
            "ppo_config": ppo_config.__dict__,
            "dqn_config": dqn_config.__dict__,
            "rl_config": rl_config.__dict__,
        },
        indent=2,
    )
)
print(f"saved to {RESULTS_PATH}")

## Summary

`data/processed/rl_ppo_checkpoint.pt` and `rl_dqn_checkpoint.pt` now hold trained weights for `ContinuousActorCritic` and `DuelingQNetwork` respectively (state dict + `RLNetworkConfig`, plain `torch.save` — not wired into `training.checkpoint`'s pretrain-model registry, since these aren't pretrain models).

See `docs/experiments.md` for the results table above transcribed with discussion, and how it bears on [H2](hypotheses.md#h2-continuous-action-spaces-improve-market-making-performance) (does C-PPO beat D-DQN?) and [H3](hypotheses.md#h3-hybrid-reward-functions-improve-risk-adjusted-performance) (does the hybrid reward keep inventory in check while still capturing spread?).

This is a **compute-budgeted** run: `n_updates=300`/`n_steps=150_000` are far below what a paper-scale training run would use, on a single seed, with no hyperparameter search — the same caveats the Attn-LOB pretraining experiments (Exp 1/1b/2) already carry, applied here to the RL phase.